# B2A External-Synthetic Validation

This publication notebook implements the complete staged validation under:

- `configs/B2A_EXTERNAL_SYNTHETIC_VALIDATION_PROTOCOL_V1_1.md`
- `configs/B2A_EXTERNAL_SYNTHETIC_IMPLEMENTATION_SPEC_V1.md`

Stage A generates the DGP, training trajectories, and queries; later sections
fit the frozen estimator, generate paired-Monte-Carlo ground truth, evaluate
the results, and compute the prespecified secondary comparisons. Stage
boundaries and frozen scientific settings are retained throughout.


In [ ]:
# Import Stage-A-only dependencies and freeze the authoritative paths and constants.
from __future__ import annotations

import hashlib
import io
import json
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROTOCOL_PATH = ROOT / "configs/B2A_EXTERNAL_SYNTHETIC_VALIDATION_PROTOCOL_V1_1.md"
SPEC_PATH = ROOT / "configs/B2A_EXTERNAL_SYNTHETIC_IMPLEMENTATION_SPEC_V1.md"
NOTEBOOK_PATH = ROOT / "notebooks/b2a_external_synthetic_validation.ipynb"
RESULTS_DIR = ROOT / "results"

DGP_PATH = RESULTS_DIR / "b2a_external_synthetic_dgp_spec_v1.json"
QUERY_PATH = RESULTS_DIR / "b2a_external_synthetic_queries_v1.csv"
TRAINING_PATH = RESULTS_DIR / "b2a_external_synthetic_training_v1.npz"
MANIFEST_PATH = RESULTS_DIR / "b2a_external_synthetic_manifest_v1.json"

assert PROTOCOL_PATH.exists()
assert SPEC_PATH.exists()
assert sys.version_info[:2] == (3, 11), sys.version

N_NODES = 20
N_ACTIONS = 8
N_TRAJECTORIES = 400
N_STATES = 120
N_ACTION_STEPS = 119
N_QUERIES = 40
N_CONTEXT_TRANSITIONS = 10
RHO = 0.50
ACTION_FLOOR = 0.03
SUPPORT_THRESHOLD = 100
ELIGIBLE_T = np.arange(11, 118, dtype=np.int64)
EXPECTED_ALIGNED_ROWS = 42_800

STREAM_NAMES = (
    "graph",
    "coefficients",
    "intervention_targets",
    "intervention_strengths",
    "action_probabilities",
    "training_initial_states",
    "training_actions",
    "training_noise",
    "query_initial_states",
    "query_context_actions",
    "query_context_noise",
    "query_sampling",
    "monte_carlo",
    "diagnostics",
)

ENV_CONFIGS = (
    {"environment_id": "EXT0", "base_seed": 1101, "p": 0.08, "sigma": 0.10, "nonlinear": False},
    {"environment_id": "EXT1", "base_seed": 2202, "p": 0.15, "sigma": 0.15, "nonlinear": False},
    {"environment_id": "EXT2", "base_seed": 3303, "p": 0.25, "sigma": 0.20, "nonlinear": False},
    {"environment_id": "EXT3", "base_seed": 4404, "p": 0.08, "sigma": 0.10, "nonlinear": True},
    {"environment_id": "EXT4", "base_seed": 5505, "p": 0.15, "sigma": 0.20, "nonlinear": True},
    {"environment_id": "EXT5", "base_seed": 6606, "p": 0.15, "sigma": 0.15, "nonlinear": False},
)

def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()

def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print("Protocol version: B2A_EXTERNAL_SYNTHETIC_VALIDATION_PROTOCOL_V1_1")
print("Implementation specification: B2A_EXTERNAL_SYNTHETIC_IMPLEMENTATION_SPEC_V1")
print("Stage boundary: A only")


In [ ]:
# Define deterministic PRNG, graph, coefficient, intervention, and action-policy construction.
class ProtocolFeasibilityError(RuntimeError):
    pass

def construct_streams(base_seed: int):
    base = np.random.SeedSequence(base_seed)
    children = base.spawn(len(STREAM_NAMES))
    streams = {
        name: np.random.Generator(np.random.PCG64(child))
        for name, child in zip(STREAM_NAMES, children, strict=True)
    }
    provenance = {
        name: {
            "entropy": int(child.entropy),
            "spawn_key": [int(value) for value in child.spawn_key],
            "pool_size": int(child.pool_size),
            "bit_generator": "numpy.random.PCG64",
        }
        for name, child in zip(STREAM_NAMES, children, strict=True)
    }
    return streams, provenance

def generate_graph(rng: np.random.Generator, edge_probability: float) -> np.ndarray:
    adjacency = np.zeros((N_NODES, N_NODES), dtype=np.int8)
    for source in range(N_NODES):
        for target in range(source + 1, N_NODES):
            adjacency[source, target] = rng.binomial(1, edge_probability)
    return adjacency

def generate_coefficients(
    rng: np.random.Generator, adjacency: np.ndarray
) -> np.ndarray:
    coefficients = np.zeros((N_NODES, N_NODES), dtype=np.float64)
    for source in range(N_NODES):
        for target in range(source + 1, N_NODES):
            if adjacency[source, target] == 1:
                magnitude = rng.uniform(0.15, 0.60)
                sign = -1.0 if rng.integers(0, 2) == 0 else 1.0
                coefficients[source, target] = sign * magnitude
    return coefficients

def generate_intervention_targets(
    rng: np.random.Generator,
    adjacency: np.ndarray,
    environment_id: str,
) -> np.ndarray:
    if environment_id == "EXT5":
        eligible_nodes = np.flatnonzero(adjacency.sum(axis=1) >= 1)
        if eligible_nodes.size < N_ACTIONS:
            raise ProtocolFeasibilityError(
                f"{environment_id}: fewer than 8 nodes have out-degree >= 1"
            )
        population = eligible_nodes
    else:
        population = np.arange(N_NODES, dtype=np.int64)
    return rng.choice(population, size=N_ACTIONS, replace=False).astype(np.int64)

def generate_intervention_strengths(rng: np.random.Generator) -> np.ndarray:
    return rng.uniform(0.20, 0.50, size=N_ACTIONS).astype(np.float64)

def generate_action_probabilities(rng: np.random.Generator):
    raw = rng.dirichlet(np.full(N_ACTIONS, 2.0, dtype=np.float64))
    remaining = 1.0 - N_ACTIONS * ACTION_FLOOR
    q = np.maximum(raw - ACTION_FLOOR, 0.0)
    if q.sum() > 0.0:
        probabilities = ACTION_FLOOR + remaining * q / q.sum()
    else:
        probabilities = np.full(N_ACTIONS, 1.0 / N_ACTIONS, dtype=np.float64)
    probabilities = probabilities.astype(np.float64)
    assert abs(float(probabilities.sum()) - 1.0) <= 1e-12
    assert float(probabilities.min()) >= ACTION_FLOOR - 1e-12
    return raw.astype(np.float64), probabilities


In [ ]:
# Define synchronous dynamics and the single canonical training/query construction path.
def synchronous_step(
    current_state: np.ndarray,
    actions: np.ndarray,
    noise: np.ndarray,
    coefficients: np.ndarray,
    intervention_targets: np.ndarray,
    intervention_strengths: np.ndarray,
    nonlinear: bool,
) -> np.ndarray:
    if nonlinear:
        parent_basis = 0.75 * current_state + 0.25 * np.square(current_state)
    else:
        parent_basis = current_state
    parent_contribution = parent_basis @ coefficients
    preactivation = RHO * current_state + parent_contribution + noise
    rows = np.arange(current_state.shape[0])
    targets = intervention_targets[actions]
    preactivation[rows, targets] += intervention_strengths[actions]
    return np.tanh(preactivation)

def generate_training_trajectories(
    streams,
    coefficients: np.ndarray,
    intervention_targets: np.ndarray,
    intervention_strengths: np.ndarray,
    action_probabilities: np.ndarray,
    sigma: float,
    nonlinear: bool,
):
    initial_states = streams["training_initial_states"].uniform(
        -0.5, 0.5, size=(N_TRAJECTORIES, N_NODES)
    )
    actions = streams["training_actions"].choice(
        N_ACTIONS,
        size=(N_TRAJECTORIES, N_ACTION_STEPS),
        p=action_probabilities,
    ).astype(np.int8)
    noise = streams["training_noise"].normal(
        0.0, sigma, size=(N_TRAJECTORIES, N_ACTION_STEPS, N_NODES)
    )
    states = np.empty(
        (N_TRAJECTORIES, N_STATES, N_NODES), dtype=np.float64
    )
    states[:, 0, :] = initial_states
    for step in range(N_ACTION_STEPS):
        states[:, step + 1, :] = synchronous_step(
            states[:, step, :],
            actions[:, step],
            noise[:, step, :],
            coefficients,
            intervention_targets,
            intervention_strengths,
            nonlinear,
        )
    return states, actions

def build_aligned_rows(states: np.ndarray, actions: np.ndarray):
    features = states[:, ELIGIBLE_T - 1, :].reshape(-1, N_NODES)
    aligned_actions = actions[:, ELIGIBLE_T].reshape(-1)
    targets = states[:, ELIGIBLE_T + 2, :].reshape(-1, N_NODES)
    trajectory_ids = np.repeat(np.arange(N_TRAJECTORIES), ELIGIBLE_T.size)
    time_indices = np.tile(ELIGIBLE_T, N_TRAJECTORIES)
    assert features.shape == (EXPECTED_ALIGNED_ROWS, N_NODES)
    assert aligned_actions.shape == (EXPECTED_ALIGNED_ROWS,)
    assert targets.shape == (EXPECTED_ALIGNED_ROWS, N_NODES)
    return {
        "features": features,
        "actions": aligned_actions,
        "targets": targets,
        "trajectory_id": trajectory_ids,
        "t": time_indices,
    }

def generate_query_contexts(
    streams,
    coefficients: np.ndarray,
    intervention_targets: np.ndarray,
    intervention_strengths: np.ndarray,
    action_probabilities: np.ndarray,
    sigma: float,
    nonlinear: bool,
):
    initial_states = streams["query_initial_states"].uniform(
        -0.5, 0.5, size=(N_QUERIES, N_NODES)
    )
    context_actions = streams["query_context_actions"].choice(
        N_ACTIONS,
        size=(N_QUERIES, N_CONTEXT_TRANSITIONS),
        p=action_probabilities,
    ).astype(np.int8)
    context_noise = streams["query_context_noise"].normal(
        0.0, sigma, size=(N_QUERIES, N_CONTEXT_TRANSITIONS, N_NODES)
    )
    current = initial_states.copy()
    for step in range(N_CONTEXT_TRANSITIONS):
        current = synchronous_step(
            current,
            context_actions[:, step],
            context_noise[:, step, :],
            coefficients,
            intervention_targets,
            intervention_strengths,
            nonlinear,
        )
    return current

def sample_queries(
    rng: np.random.Generator,
    environment_id: str,
    conditioning_states: np.ndarray,
    supported_actions: np.ndarray,
    adjacency: np.ndarray,
    intervention_targets: np.ndarray,
):
    records = []
    if environment_id == "EXT5":
        eligible_actions = np.array(
            [
                action
                for action in supported_actions
                if adjacency[intervention_targets[action]].sum() >= 1
            ],
            dtype=np.int64,
        )
        if eligible_actions.size < 2:
            raise ProtocolFeasibilityError(
                f"{environment_id}: fewer than two supported mediation-eligible actions"
            )
    else:
        eligible_actions = supported_actions.astype(np.int64)

    for query_id in range(N_QUERIES):
        intervention_action = int(rng.choice(eligible_actions))
        reference_pool = eligible_actions[eligible_actions != intervention_action]
        reference_action = int(rng.choice(reference_pool))

        if environment_id == "EXT5":
            intervention_target = int(intervention_targets[intervention_action])
            reference_target = int(intervention_targets[reference_action])
            children = set(np.flatnonzero(adjacency[intervention_target]).tolist())
            children.update(np.flatnonzero(adjacency[reference_target]).tolist())
            children.discard(intervention_target)
            children.discard(reference_target)
            candidate_y = np.array(sorted(children), dtype=np.int64)
            if candidate_y.size == 0:
                raise ProtocolFeasibilityError(
                    f"{environment_id}: query {query_id} has empty candidate_Y"
                )
            target_construct = int(rng.choice(candidate_y))
        else:
            target_construct = int(rng.integers(0, N_NODES))

        record = {
            "environment_id": environment_id,
            "query_id": query_id,
        }
        record.update(
            {
                f"x_{node:02d}": float(conditioning_states[query_id, node])
                for node in range(N_NODES)
            }
        )
        record.update(
            {
                "intervention_action": intervention_action,
                "reference_action": reference_action,
                "target_construct": target_construct,
            }
        )
        records.append(record)
    return pd.DataFrame.from_records(records), eligible_actions


In [ ]:
# Assemble all six Stage-A environments and their non-effect diagnostics in one pipeline.
def environment_diagnostics(
    states: np.ndarray,
    adjacency: np.ndarray,
    support_counts: np.ndarray,
    queries: pd.DataFrame,
):
    construct_variances = states.reshape(-1, N_NODES).var(axis=0, ddof=0)
    duplicate_count = int(
        queries.duplicated(
            subset=["intervention_action", "reference_action", "target_construct"],
            keep="first",
        ).sum()
    )
    edge_count = int(adjacency.sum())
    return {
        "realized_edge_count": edge_count,
        "realized_graph_density": edge_count / (N_NODES * (N_NODES - 1) / 2),
        "aligned_action_support_counts": [int(value) for value in support_counts],
        "number_of_supported_actions": int((support_counts >= SUPPORT_THRESHOLD).sum()),
        "state_saturation_fraction_abs_gt_0_95": float(
            np.mean(np.abs(states) > 0.95)
        ),
        "mean_state_variance_across_constructs": float(construct_variances.mean()),
        "minimum_construct_variance": float(construct_variances.min()),
        "maximum_construct_variance": float(construct_variances.max()),
        "query_count": int(len(queries)),
        "query_duplicate_iry_count_beyond_first": duplicate_count,
        "query_unique_iry_count": int(len(queries) - duplicate_count),
    }

def generate_stage_a():
    environment_specs = []
    training_arrays = {}
    query_frames = []

    for config in ENV_CONFIGS:
        environment_id = config["environment_id"]
        streams, stream_provenance = construct_streams(config["base_seed"])

        adjacency = generate_graph(streams["graph"], config["p"])
        coefficients = generate_coefficients(streams["coefficients"], adjacency)
        intervention_targets = generate_intervention_targets(
            streams["intervention_targets"], adjacency, environment_id
        )
        intervention_strengths = generate_intervention_strengths(
            streams["intervention_strengths"]
        )
        raw_action_probabilities, action_probabilities = (
            generate_action_probabilities(streams["action_probabilities"])
        )

        states, actions = generate_training_trajectories(
            streams,
            coefficients,
            intervention_targets,
            intervention_strengths,
            action_probabilities,
            config["sigma"],
            config["nonlinear"],
        )
        aligned = build_aligned_rows(states, actions)
        support_counts = np.bincount(
            aligned["actions"], minlength=N_ACTIONS
        ).astype(np.int64)
        supported_actions = np.flatnonzero(
            support_counts >= SUPPORT_THRESHOLD
        ).astype(np.int64)

        conditioning_states = generate_query_contexts(
            streams,
            coefficients,
            intervention_targets,
            intervention_strengths,
            action_probabilities,
            config["sigma"],
            config["nonlinear"],
        )
        queries, query_eligible_actions = sample_queries(
            streams["query_sampling"],
            environment_id,
            conditioning_states,
            supported_actions,
            adjacency,
            intervention_targets,
        )
        diagnostics = environment_diagnostics(
            states, adjacency, support_counts, queries
        )

        ext5_eligible_target_nodes = (
            np.flatnonzero(adjacency.sum(axis=1) >= 1).astype(np.int64)
            if environment_id == "EXT5"
            else np.array([], dtype=np.int64)
        )
        mediation_eligible_actions = (
            query_eligible_actions
            if environment_id == "EXT5"
            else np.array([], dtype=np.int64)
        )

        environment_specs.append(
            {
                "environment_id": environment_id,
                "base_seed": int(config["base_seed"]),
                "component_seed_sequences": stream_provenance,
                "edge_probability_p": float(config["p"]),
                "noise_sigma": float(config["sigma"]),
                "adjacency_convention": "adjacency[i,j] = 1 means i -> j",
                "adjacency": adjacency.astype(int).tolist(),
                "coefficient_matrix": coefficients.tolist(),
                "intervention_target_by_action": intervention_targets.tolist(),
                "intervention_strength_by_action": intervention_strengths.tolist(),
                "raw_dirichlet_action_probabilities": raw_action_probabilities.tolist(),
                "action_probabilities": action_probabilities.tolist(),
                "support_counts_from_aligned_rows": support_counts.tolist(),
                "supported_actions": supported_actions.tolist(),
                "ext5_eligible_target_nodes": ext5_eligible_target_nodes.tolist(),
                "ext5_mediation_eligible_actions": mediation_eligible_actions.tolist(),
                "diagnostics": diagnostics,
                "mechanism": {
                    "node_count": N_NODES,
                    "action_count": N_ACTIONS,
                    "rho": RHO,
                    "state_update": "synchronous",
                    "state_transform": "tanh",
                    "initial_state_distribution": "independent Uniform(-0.5, 0.5)",
                    "coefficient_magnitude_distribution": "Uniform(0.15, 0.60)",
                    "coefficient_sign_distribution": "equiprobable -1 or +1",
                    "intervention_strength_distribution": "Uniform(0.20, 0.50)",
                    "action_probability_distribution": "Dirichlet([2,2,2,2,2,2,2,2])",
                    "action_probability_floor": ACTION_FLOOR,
                    "nonlinear": bool(config["nonlinear"]),
                    "nonlinear_parental_term": (
                        "0.75 * beta * X + 0.25 * beta * X^2"
                        if config["nonlinear"]
                        else None
                    ),
                },
                "training": {
                    "states_shape": [N_TRAJECTORIES, N_STATES, N_NODES],
                    "actions_shape": [N_TRAJECTORIES, N_ACTION_STEPS],
                    "eligible_t_inclusive": [11, 117],
                    "aligned_rows": EXPECTED_ALIGNED_ROWS,
                    "ordering": "trajectory_id ascending, then t ascending",
                    "states_npz_key": f"{environment_id}_states",
                    "actions_npz_key": f"{environment_id}_actions",
                },
                "queries": {
                    "query_count": N_QUERIES,
                    "conditioning_state": "X_10 after 10 independent context transitions",
                    "query_key": ["environment_id", "query_id"],
                    "context_partition": "separate PRNG streams; never used for training",
                },
            }
        )
        training_arrays[f"{environment_id}_states"] = states
        training_arrays[f"{environment_id}_actions"] = actions
        query_frames.append(queries)

    query_table = pd.concat(query_frames, ignore_index=True)
    dgp_spec = {
        "schema_version": "b2a_external_synthetic_dgp_spec_v1",
        "stage": "A",
        "protocol_version": "B2A_EXTERNAL_SYNTHETIC_VALIDATION_PROTOCOL_V1_1",
        "implementation_spec_version": "B2A_EXTERNAL_SYNTHETIC_IMPLEMENTATION_SPEC_V1",
        "runtime": {
            "python": platform.python_version(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "prng": "numpy.random.PCG64",
            "seed_derivation": "numpy.random.SeedSequence.spawn",
        },
        "training_storage": {
            "format": "compressed NPZ",
            "allow_pickle": False,
            "keys": list(training_arrays.keys()),
            "states_dtype": "float64",
            "actions_dtype": "int8",
        },
        "environments": environment_specs,
    }
    return {
        "dgp_spec": dgp_spec,
        "queries": query_table,
        "training_arrays": training_arrays,
    }

def json_payload(value) -> bytes:
    return (
        json.dumps(value, sort_keys=True, indent=2, allow_nan=False) + "\n"
    ).encode("utf-8")

def csv_payload(frame: pd.DataFrame) -> bytes:
    buffer = io.StringIO()
    frame.to_csv(
        buffer,
        index=False,
        lineterminator="\n",
        float_format=lambda value: repr(float(value)),
    )
    return buffer.getvalue().encode("utf-8")

def npz_payload(arrays) -> bytes:
    buffer = io.BytesIO()
    np.savez_compressed(buffer, **arrays)
    return buffer.getvalue()

def canonical_json_bytes(value) -> bytes:
    return json.dumps(
        value, sort_keys=True, separators=(",", ":"), allow_nan=False
    ).encode("utf-8")

def training_scientific_sha256(arrays) -> str:
    digest = hashlib.sha256()
    for key in sorted(arrays):
        array = np.ascontiguousarray(arrays[key])
        key_bytes = key.encode("utf-8")
        dtype_bytes = array.dtype.str.encode("ascii")
        shape_bytes = canonical_json_bytes(list(array.shape))
        for token in (key_bytes, dtype_bytes, shape_bytes):
            digest.update(len(token).to_bytes(8, "big"))
            digest.update(token)
        digest.update(array.tobytes(order="C"))
    return digest.hexdigest()

def stage_a_scientific_hashes(stage_a, payloads):
    return {
        DGP_PATH.relative_to(ROOT).as_posix(): sha256_bytes(
            canonical_json_bytes({"environments": stage_a["dgp_spec"]["environments"]})
        ),
        QUERY_PATH.relative_to(ROOT).as_posix(): sha256_bytes(payloads[QUERY_PATH]),
        TRAINING_PATH.relative_to(ROOT).as_posix(): training_scientific_sha256(
            stage_a["training_arrays"]
        ),
    }

def canonical_payloads(stage_a):
    return {
        DGP_PATH: json_payload(stage_a["dgp_spec"]),
        QUERY_PATH: csv_payload(stage_a["queries"]),
        TRAINING_PATH: npz_payload(stage_a["training_arrays"]),
    }


In [ ]:
# Generate Stage A twice in memory and require deterministic equality before any canonical write.
stage_a = generate_stage_a()
stage_a_repeat = generate_stage_a()

assert stage_a["dgp_spec"] == stage_a_repeat["dgp_spec"]
assert stage_a["queries"].equals(stage_a_repeat["queries"])
assert stage_a["training_arrays"].keys() == stage_a_repeat["training_arrays"].keys()
for key in stage_a["training_arrays"]:
    assert np.array_equal(
        stage_a["training_arrays"][key],
        stage_a_repeat["training_arrays"][key],
        equal_nan=True,
    )

canonical_bytes = canonical_payloads(stage_a)
repeat_bytes = canonical_payloads(stage_a_repeat)
deterministic_hashes = stage_a_scientific_hashes(stage_a, canonical_bytes)
repeat_hashes = stage_a_scientific_hashes(stage_a_repeat, repeat_bytes)
assert deterministic_hashes == repeat_hashes

del stage_a_repeat
del repeat_bytes

print("Deterministic in-memory regeneration: PASS")
for relative_path, digest in deterministic_hashes.items():
    print(f"{relative_path}: {digest}")


In [ ]:
# Verify every frozen Stage-A integrity and leakage-boundary assertion before persistence.
expected_environment_ids = [f"EXT{index}" for index in range(6)]
actual_environment_ids = [
    environment["environment_id"] for environment in stage_a["dgp_spec"]["environments"]
]
assert actual_environment_ids == expected_environment_ids
assert set(stage_a["training_arrays"]) == {
    f"EXT{index}_{kind}"
    for index in range(6)
    for kind in ("states", "actions")
}

query_table = stage_a["queries"]
assert query_table.shape == (240, 25)
assert query_table[["environment_id", "query_id"]].duplicated().sum() == 0
assert not query_table.isna().any().any()
assert (query_table["intervention_action"] != query_table["reference_action"]).all()

diagnostic_rows = []
for environment in stage_a["dgp_spec"]["environments"]:
    environment_id = environment["environment_id"]
    adjacency = np.asarray(environment["adjacency"], dtype=np.int8)
    coefficients = np.asarray(environment["coefficient_matrix"], dtype=np.float64)
    targets = np.asarray(environment["intervention_target_by_action"], dtype=np.int64)
    probabilities = np.asarray(environment["action_probabilities"], dtype=np.float64)
    support_counts = np.asarray(
        environment["support_counts_from_aligned_rows"], dtype=np.int64
    )
    supported_actions = set(environment["supported_actions"])
    states = stage_a["training_arrays"][f"{environment_id}_states"]
    actions = stage_a["training_arrays"][f"{environment_id}_actions"]
    environment_queries = query_table.loc[
        query_table["environment_id"] == environment_id
    ].sort_values("query_id")

    assert adjacency.shape == (N_NODES, N_NODES)
    assert np.array_equal(adjacency, np.triu(adjacency, k=1))
    assert coefficients.shape == adjacency.shape
    assert np.all(coefficients[adjacency == 0] == 0.0)
    assert np.all(coefficients[adjacency == 1] != 0.0)
    assert targets.shape == (N_ACTIONS,)
    assert np.unique(targets).size == N_ACTIONS
    assert abs(float(probabilities.sum()) - 1.0) <= 1e-12
    assert float(probabilities.min()) >= ACTION_FLOOR - 1e-12
    assert states.shape == (N_TRAJECTORIES, N_STATES, N_NODES)
    assert states.dtype == np.float64
    assert actions.shape == (N_TRAJECTORIES, N_ACTION_STEPS)
    assert actions.dtype == np.int8

    aligned = build_aligned_rows(states, actions)
    assert aligned["features"].shape[0] == EXPECTED_ALIGNED_ROWS
    independent_counts = np.bincount(
        aligned["actions"], minlength=N_ACTIONS
    )
    assert np.array_equal(independent_counts, support_counts)
    assert len(environment_queries) == N_QUERIES
    assert environment_queries["query_id"].tolist() == list(range(N_QUERIES))
    assert set(environment_queries["intervention_action"]).issubset(supported_actions)
    assert set(environment_queries["reference_action"]).issubset(supported_actions)

    training_spawn_keys = {
        tuple(environment["component_seed_sequences"][name]["spawn_key"])
        for name in (
            "training_initial_states",
            "training_actions",
            "training_noise",
        )
    }
    query_spawn_keys = {
        tuple(environment["component_seed_sequences"][name]["spawn_key"])
        for name in (
            "query_initial_states",
            "query_context_actions",
            "query_context_noise",
        )
    }
    assert training_spawn_keys.isdisjoint(query_spawn_keys)

    ext5_status = "not applicable"
    if environment_id == "EXT5":
        eligible_targets = np.flatnonzero(adjacency.sum(axis=1) >= 1)
        assert eligible_targets.size >= N_ACTIONS
        assert set(targets).issubset(set(eligible_targets))
        mediation_actions = set(environment["ext5_mediation_eligible_actions"])
        assert len(mediation_actions) >= 2
        for row in environment_queries.itertuples(index=False):
            assert row.intervention_action in mediation_actions
            assert row.reference_action in mediation_actions
            target_i = targets[row.intervention_action]
            target_r = targets[row.reference_action]
            candidate_y = set(np.flatnonzero(adjacency[target_i]).tolist())
            candidate_y.update(np.flatnonzero(adjacency[target_r]).tolist())
            candidate_y.discard(int(target_i))
            candidate_y.discard(int(target_r))
            assert row.target_construct in candidate_y
        ext5_status = "PASS"

    diagnostics = environment["diagnostics"]
    diagnostic_rows.append(
        {
            "environment_id": environment_id,
            "edge_count": diagnostics["realized_edge_count"],
            "edge_density": diagnostics["realized_graph_density"],
            "supported_actions": diagnostics["number_of_supported_actions"],
            "saturation_fraction": diagnostics[
                "state_saturation_fraction_abs_gt_0_95"
            ],
            "query_duplicate_iry_beyond_first": diagnostics[
                "query_duplicate_iry_count_beyond_first"
            ],
            "ext5_feasibility": ext5_status,
        }
    )

for forbidden_column in (
    "tau" + "_gt",
    "prediction",
    "ground" + "_truth",
    "mae",
    "rmse",
    "pearson",
    "spearman",
    "sign_agreement",
):
    assert forbidden_column not in {column.lower() for column in query_table.columns}
    assert forbidden_column not in json.dumps(
        stage_a["dgp_spec"], sort_keys=True
    ).lower()

diagnostic_table = pd.DataFrame(diagnostic_rows)
print("Stage-A integrity assertions: PASS")
display(diagnostic_table)


In [ ]:
# Persist only the validated Stage-A artifacts and create their non-self-referential manifest.
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for path, payload in canonical_bytes.items():
    path.write_bytes(payload)

artifact_records = []
for path in (DGP_PATH, QUERY_PATH, TRAINING_PATH):
    artifact_records.append(
        {
            "relative_path": path.relative_to(ROOT).as_posix(),
            "scientific_payload_sha256": deterministic_hashes[
                path.relative_to(ROOT).as_posix()
            ],
            "byte_size": path.stat().st_size,
            "stage": "A",
        }
    )

manifest = {
    "schema_version": "b2a_external_synthetic_manifest_v1",
    "stage": "A",
    "protocol_version": "B2A_EXTERNAL_SYNTHETIC_VALIDATION_PROTOCOL_V1_1",
    "implementation_spec_version": "B2A_EXTERNAL_SYNTHETIC_IMPLEMENTATION_SPEC_V1",
    "artifacts": artifact_records,
    "self_hash_included": False,
}
MANIFEST_PATH.write_bytes(json_payload(manifest))
manifest_sha256 = sha256_file(MANIFEST_PATH)

print("Stage-A canonical artifacts written:")
for record in artifact_records:
    print(
        f"- {record['relative_path']}: "
        f"{record['byte_size']} bytes, scientific_payload_sha256="
        f"{record['scientific_payload_sha256']}"
    )
print(
    f"- {MANIFEST_PATH.relative_to(ROOT).as_posix()}: "
    f"{MANIFEST_PATH.stat().st_size} bytes, sha256={manifest_sha256}"
)


In [ ]:
# Re-open persisted artifacts and enforce the final Stage-A stop and leakage boundaries.
for path, expected_payload in canonical_bytes.items():
    assert path.read_bytes() == expected_payload

loaded_manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
assert loaded_manifest["stage"] == "A"
assert loaded_manifest["self_hash_included"] is False
assert len(loaded_manifest["artifacts"]) == 3
for record in loaded_manifest["artifacts"]:
    artifact_path = ROOT / record["relative_path"]
    assert artifact_path.exists()
    assert artifact_path.stat().st_size == record["byte_size"]
    assert record["scientific_payload_sha256"] == deterministic_hashes[
        record["relative_path"]
    ]

for prohibited_path in (
    RESULTS_DIR / "b2a_external_synthetic_predictions_v1.csv",
    RESULTS_DIR / "b2a_external_synthetic_ground_truth_v1.csv",
    RESULTS_DIR / "b2a_external_synthetic_summary_v1.csv",
    RESULTS_DIR / "b2a_external_synthetic_environment_summary_v1.csv",
):
    assert not prohibited_path.exists(), prohibited_path

with np.load(TRAINING_PATH, allow_pickle=False) as persisted_training:
    assert persisted_training.files == list(stage_a["training_arrays"].keys())
    for key in persisted_training.files:
        assert np.array_equal(persisted_training[key], stage_a["training_arrays"][key])

persisted_queries = pd.read_csv(QUERY_PATH)
assert persisted_queries.shape == (240, 25)
assert persisted_queries[["environment_id", "query_id"]].duplicated().sum() == 0

print("Persisted-artifact verification: PASS")
print("Leakage boundary: PASS")
print("STOP: Stage A complete; Stages B, C, D, and E were not executed.")


## Stage B — Frozen B2A Prediction Generation

This section executes Stage B only. It verifies the frozen Stage A inputs,
reconstructs the specified aligned rows, fits the unchanged action-specific
Ridge estimator, and freezes 240 predictions. It does not construct or inspect
intervention-effect ground truth and does not evaluate prediction performance.


In [ ]:
# Verify frozen Stage A hashes and load only the inputs permitted for blind prediction.
import ast
import hashlib
import io
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

STAGE_B_ROOT = Path.cwd()
if STAGE_B_ROOT.name == "notebooks":
    STAGE_B_ROOT = STAGE_B_ROOT.parent

STAGE_A_DGP_PATH = STAGE_B_ROOT / "results/b2a_external_synthetic_dgp_spec_v1.json"
STAGE_A_QUERY_PATH = STAGE_B_ROOT / "results/b2a_external_synthetic_queries_v1.csv"
STAGE_A_TRAINING_PATH = STAGE_B_ROOT / "results/b2a_external_synthetic_training_v1.npz"
STAGE_A_MANIFEST_PATH = STAGE_B_ROOT / "results/b2a_external_synthetic_manifest_v1.json"
STAGE_B_PREDICTION_PATH = STAGE_B_ROOT / "results/b2a_external_synthetic_predictions_v1.csv"
STAGE_B_NOTEBOOK_PATH = STAGE_B_ROOT / "notebooks/b2a_external_synthetic_validation.ipynb"

FROZEN_STAGE_A_SCIENTIFIC_HASHES = {
    "results/b2a_external_synthetic_dgp_spec_v1.json":
        "95884a97bda259755d751ac597b172c0ec17878d0767ad7279bb61df39b7cca0",
    "results/b2a_external_synthetic_queries_v1.csv":
        "d0c022193284c495751b7168d62db8206a397f02537e54ca2e165a0248b83d6b",
    "results/b2a_external_synthetic_training_v1.npz":
        "f215ab76c2106a0971f9353630b47c430477c105e858cf79adaff6fb806bd838",
}

def stage_b_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def stage_b_canonical_json_bytes(value) -> bytes:
    return json.dumps(
        value, sort_keys=True, separators=(",", ":"), allow_nan=False
    ).encode("utf-8")

def stage_b_dgp_scientific_sha256(path: Path) -> str:
    dgp = json.loads(path.read_text(encoding="utf-8"))
    return hashlib.sha256(
        stage_b_canonical_json_bytes({"environments": dgp["environments"]})
    ).hexdigest()

def stage_b_training_scientific_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with np.load(path, allow_pickle=False) as arrays:
        for key in sorted(arrays.files):
            array = np.ascontiguousarray(arrays[key])
            tokens = (
                key.encode("utf-8"),
                array.dtype.str.encode("ascii"),
                stage_b_canonical_json_bytes(list(array.shape)),
            )
            for token in tokens:
                digest.update(len(token).to_bytes(8, "big"))
                digest.update(token)
            digest.update(array.tobytes(order="C"))
    return digest.hexdigest()

def stage_b_scientific_sha256(relative_path: str) -> str:
    path = STAGE_B_ROOT / relative_path
    if path == STAGE_A_DGP_PATH:
        return stage_b_dgp_scientific_sha256(path)
    if path == STAGE_A_TRAINING_PATH:
        return stage_b_training_scientific_sha256(path)
    return stage_b_sha256(path)

stage_a_manifest = json.loads(STAGE_A_MANIFEST_PATH.read_text(encoding="utf-8"))
manifest_records = {
    record["relative_path"]: record for record in stage_a_manifest["artifacts"]
}
assert stage_a_manifest["stage"] == "A"
assert stage_a_manifest["schema_version"] == "b2a_external_synthetic_manifest_v1"
assert set(manifest_records) == set(FROZEN_STAGE_A_SCIENTIFIC_HASHES)
for relative_path, expected_hash in FROZEN_STAGE_A_SCIENTIFIC_HASHES.items():
    artifact_path = STAGE_B_ROOT / relative_path
    assert artifact_path.exists()
    assert stage_b_scientific_sha256(relative_path) == expected_hash
    record = manifest_records[relative_path]
    assert record["scientific_payload_sha256"] == expected_hash
    assert record["byte_size"] == artifact_path.stat().st_size
    assert record["stage"] == "A"

expected_query_columns = (
    ["environment_id", "query_id"]
    + [f"x_{node:02d}" for node in range(20)]
    + ["intervention_action", "reference_action", "target_construct"]
)
stage_b_queries = pd.read_csv(STAGE_A_QUERY_PATH, float_precision="round_trip")
assert stage_b_queries.columns.tolist() == expected_query_columns
assert stage_b_queries.shape == (240, 25)
assert not stage_b_queries.isna().any().any()

stage_b_training = np.load(STAGE_A_TRAINING_PATH, allow_pickle=False)
assert stage_b_training.files == [
    key
    for environment_index in range(6)
    for key in (
        f"EXT{environment_index}_states",
        f"EXT{environment_index}_actions",
    )
]
assert not STAGE_B_PREDICTION_PATH.exists()
for prohibited_future_path in (
    STAGE_B_ROOT / "results/b2a_external_synthetic_ground_truth_v1.csv",
    STAGE_B_ROOT / "results/b2a_external_synthetic_summary_v1.csv",
    STAGE_B_ROOT / "results/b2a_external_synthetic_environment_summary_v1.csv",
):
    assert not prohibited_future_path.exists()

print("Frozen Stage A input hashes: PASS")
print("Queries and training inputs loaded without modification.")


In [ ]:
# Reconstruct frozen aligned rows and fit one unchanged Ridge model per action and environment.
STAGE_B_ENVIRONMENTS = [f"EXT{index}" for index in range(6)]
STAGE_B_ELIGIBLE_T = np.arange(11, 118, dtype=np.int64)
STAGE_B_EXPECTED_ROWS = 42_800
STAGE_B_SUPPORT_THRESHOLD = 100

def fit_frozen_b2a_models(training_archive):
    fitted_models = {}
    fit_records = []
    for environment_id in STAGE_B_ENVIRONMENTS:
        states = training_archive[f"{environment_id}_states"]
        actions = training_archive[f"{environment_id}_actions"]
        assert states.shape == (400, 120, 20)
        assert actions.shape == (400, 119)
        features = states[:, STAGE_B_ELIGIBLE_T - 1, :].reshape(-1, 20)
        aligned_actions = actions[:, STAGE_B_ELIGIBLE_T].reshape(-1)
        targets = states[:, STAGE_B_ELIGIBLE_T + 2, :].reshape(-1, 20)
        assert features.shape == (STAGE_B_EXPECTED_ROWS, 20)
        assert aligned_actions.shape == (STAGE_B_EXPECTED_ROWS,)
        assert targets.shape == (STAGE_B_EXPECTED_ROWS, 20)

        environment_models = {}
        for action in range(8):
            action_mask = aligned_actions == action
            n_training_rows = int(action_mask.sum())
            assert n_training_rows >= STAGE_B_SUPPORT_THRESHOLD
            model = Ridge(alpha=1.0)
            model.fit(features[action_mask], targets[action_mask])
            assert model.alpha == 1.0
            assert model.fit_intercept is True
            assert model.solver == "auto"
            assert model.coef_.shape == (20, 20)
            assert model.intercept_.shape == (20,)
            environment_models[action] = model
            fit_records.append({
                "environment_id": environment_id,
                "action": action,
                "n_training_rows": n_training_rows,
                "coefficient_shape": list(model.coef_.shape),
                "intercept_shape": list(model.intercept_.shape),
            })
        fitted_models[environment_id] = environment_models
    return fitted_models, pd.DataFrame(fit_records)

stage_b_models, stage_b_fit_table = fit_frozen_b2a_models(stage_b_training)
assert stage_b_fit_table.shape == (48, 5)
print("Frozen B2A fits complete: 48 action-specific models.")
display(stage_b_fit_table)


In [ ]:
# Generate all blind query predictions and prediction-only diagnostics from permitted inputs.
STAGE_B_STATE_COLUMNS = [f"x_{node:02d}" for node in range(20)]
STAGE_B_PREDICTION_COLUMNS = [
    "environment_id", "query_id", "intervention_action", "reference_action",
    "target", "pred_intervention", "pred_reference", "tau_hat",
]

def generate_stage_b_predictions(models, query_table):
    records = []
    ordered_queries = query_table.sort_values(
        ["environment_id", "query_id"]
    ).reset_index(drop=True)
    for _, query in ordered_queries.iterrows():
        environment_id = str(query["environment_id"])
        query_id = int(query["query_id"])
        intervention_action = int(query["intervention_action"])
        reference_action = int(query["reference_action"])
        target = int(query["target_construct"])
        conditioning_state = query[STAGE_B_STATE_COLUMNS].to_numpy(
            dtype=np.float64
        ).reshape(1, -1)
        pred_intervention_vector = models[environment_id][
            intervention_action
        ].predict(conditioning_state)[0]
        pred_reference_vector = models[environment_id][
            reference_action
        ].predict(conditioning_state)[0]
        pred_intervention = float(pred_intervention_vector[target])
        pred_reference = float(pred_reference_vector[target])
        records.append({
            "environment_id": environment_id,
            "query_id": query_id,
            "intervention_action": intervention_action,
            "reference_action": reference_action,
            "target": target,
            "pred_intervention": pred_intervention,
            "pred_reference": pred_reference,
            "tau_hat": pred_intervention - pred_reference,
        })
    return pd.DataFrame.from_records(records, columns=STAGE_B_PREDICTION_COLUMNS)

stage_b_predictions = generate_stage_b_predictions(stage_b_models, stage_b_queries)
assert stage_b_predictions.shape == (240, 8)
assert stage_b_predictions.groupby("environment_id", sort=False).size().tolist() == [40] * 6
assert stage_b_predictions[["environment_id", "query_id"]].duplicated().sum() == 0
numeric_prediction_columns = ["pred_intervention", "pred_reference", "tau_hat"]
assert np.isfinite(stage_b_predictions[numeric_prediction_columns].to_numpy()).all()

expected_mapping = (
    stage_b_queries.sort_values(["environment_id", "query_id"])[
        ["environment_id", "query_id", "intervention_action",
         "reference_action", "target_construct"]
    ]
    .rename(columns={"target_construct": "target"})
    .reset_index(drop=True)
)
pd.testing.assert_frame_equal(
    stage_b_predictions[expected_mapping.columns],
    expected_mapping,
    check_dtype=False,
)

prediction_only_diagnostics = (
    stage_b_predictions.groupby("environment_id", sort=False)
    .agg(
        prediction_count=("tau_hat", "size"),
        tau_hat_min=("tau_hat", "min"),
        tau_hat_max=("tau_hat", "max"),
        tau_hat_mean=("tau_hat", "mean"),
        tau_hat_std=("tau_hat", lambda values: values.std(ddof=0)),
        pred_intervention_min=("pred_intervention", "min"),
        pred_intervention_max=("pred_intervention", "max"),
        pred_reference_min=("pred_reference", "min"),
        pred_reference_max=("pred_reference", "max"),
    )
    .reset_index()
)
assert prediction_only_diagnostics["prediction_count"].tolist() == [40] * 6
print("Prediction-only integrity assertions: PASS")
display(prediction_only_diagnostics)


In [ ]:
# Audit Stage B code before saving so no future-stage computation crosses the blind boundary.
stage_b_notebook = json.loads(STAGE_B_NOTEBOOK_PATH.read_text(encoding="utf-8"))
stage_b_cell_ids = {
    "stage-b-inputs", "stage-b-fit", "stage-b-predict",
    "stage-b-leakage", "stage-b-freeze", "stage-b-final",
}
stage_b_code = "\n".join(
    "".join(cell.get("source", []))
    for cell in stage_b_notebook["cells"]
    if cell.get("id") in stage_b_cell_ids
)
stage_b_tree = ast.parse(stage_b_code)
stage_b_imports = set()
stage_b_called_attributes = set()
stage_b_function_names = set()
for node in ast.walk(stage_b_tree):
    if isinstance(node, ast.Import):
        stage_b_imports.update(alias.name for alias in node.names)
    elif isinstance(node, ast.ImportFrom):
        stage_b_imports.add(node.module or "")
    elif isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute):
        stage_b_called_attributes.add(node.func.attr)
    elif isinstance(node, ast.FunctionDef):
        stage_b_function_names.add(node.name.lower())

assert not any(
    module.startswith(("scipy.stats", "statsmodels"))
    for module in stage_b_imports
)
for forbidden_call in (
    "mean_absolute_error", "mean_squared_error", "root_mean_squared_error",
    "pearsonr", "spearmanr",
):
    assert forbidden_call not in stage_b_called_attributes
assert not any(
    "monte_carlo" in name or "ground_truth" in name
    for name in stage_b_function_names
)
assert "tau" + "_gt" not in stage_b_code.lower()
assert ".fit(" in stage_b_code
assert ".predict(" in stage_b_code
stage_b_dgp_access_calls = [
    node
    for node in ast.walk(stage_b_tree)
    if isinstance(node, ast.Call)
    and isinstance(node.func, ast.Attribute)
    and isinstance(node.func.value, ast.Name)
    and node.func.value.id == "STAGE_A_DGP_PATH"
]
assert not stage_b_dgp_access_calls
assert not STAGE_B_PREDICTION_PATH.exists()

print("Pre-save Stage B blindness/leakage audit: PASS")
print("DGP artifact was hash-checked only and was not parsed for prediction.")


In [ ]:
# Refit and regenerate in memory, then freeze the byte-deterministic canonical prediction table.
def stage_b_csv_payload(frame: pd.DataFrame) -> bytes:
    buffer = io.StringIO()
    frame.to_csv(
        buffer,
        index=False,
        lineterminator="\n",
        float_format=lambda value: repr(float(value)),
    )
    return buffer.getvalue().encode("utf-8")

stage_b_repeat_models, stage_b_repeat_fit_table = fit_frozen_b2a_models(stage_b_training)
stage_b_repeat_predictions = generate_stage_b_predictions(
    stage_b_repeat_models, stage_b_queries
)
pd.testing.assert_frame_equal(
    stage_b_predictions, stage_b_repeat_predictions, check_exact=True
)
pd.testing.assert_frame_equal(
    stage_b_fit_table, stage_b_repeat_fit_table, check_exact=True
)
stage_b_prediction_bytes = stage_b_csv_payload(stage_b_predictions)
stage_b_repeat_bytes = stage_b_csv_payload(stage_b_repeat_predictions)
assert stage_b_prediction_bytes == stage_b_repeat_bytes

STAGE_B_PREDICTION_PATH.write_bytes(stage_b_prediction_bytes)
stage_b_prediction_sha256 = stage_b_sha256(STAGE_B_PREDICTION_PATH)
stage_b_prediction_byte_size = STAGE_B_PREDICTION_PATH.stat().st_size

print("Independent in-memory Stage B regeneration: PASS")
print(f"Prediction rows: {len(stage_b_predictions)}")
print(f"Prediction bytes: {stage_b_prediction_byte_size}")
print(f"Prediction SHA-256: {stage_b_prediction_sha256}")


In [ ]:
# Re-open the prediction artifact and enforce the final Stage B stop boundary.
persisted_stage_b_predictions = pd.read_csv(
    STAGE_B_PREDICTION_PATH, float_precision="round_trip"
)
assert persisted_stage_b_predictions.columns.tolist() == STAGE_B_PREDICTION_COLUMNS
assert persisted_stage_b_predictions.shape == (240, 8)
assert persisted_stage_b_predictions[["environment_id", "query_id"]].duplicated().sum() == 0
assert np.isfinite(
    persisted_stage_b_predictions[numeric_prediction_columns].to_numpy()
).all()
pd.testing.assert_frame_equal(
    persisted_stage_b_predictions,
    stage_b_predictions,
    check_exact=True,
    check_dtype=False,
)
assert stage_b_sha256(STAGE_B_PREDICTION_PATH) == stage_b_prediction_sha256

for prohibited_future_path in (
    STAGE_B_ROOT / "results/b2a_external_synthetic_ground_truth_v1.csv",
    STAGE_B_ROOT / "results/b2a_external_synthetic_summary_v1.csv",
    STAGE_B_ROOT / "results/b2a_external_synthetic_environment_summary_v1.csv",
):
    assert not prohibited_future_path.exists()

stage_b_training.close()
print("Persisted Stage B prediction verification: PASS")
print("STOP: Stage B complete; Stages C, D, and E were not executed.")


## Stage D/E — Ground Truth and Frozen Evaluation

Stage D generates the frozen paired-Monte-Carlo intervention-effect ground
truth only after verifying the frozen prediction hash. Stage E begins only
after that ground-truth file is finalized and hashed, then evaluates the frozen
predictions without tuning, estimator modification, or additional experiments.


In [ ]:
# Verify the frozen prediction boundary and load the Stage D simulator/query inputs.
from __future__ import annotations

import hashlib
import io
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

STAGE_DE_ROOT = Path.cwd()
if STAGE_DE_ROOT.name == "notebooks":
    STAGE_DE_ROOT = STAGE_DE_ROOT.parent

STAGE_DE_DGP_PATH = STAGE_DE_ROOT / "results/b2a_external_synthetic_dgp_spec_v1.json"
STAGE_DE_QUERY_PATH = STAGE_DE_ROOT / "results/b2a_external_synthetic_queries_v1.csv"
STAGE_DE_PREDICTION_PATH = STAGE_DE_ROOT / "results/b2a_external_synthetic_predictions_v1.csv"
STAGE_D_GT_PATH = STAGE_DE_ROOT / "results/b2a_external_synthetic_ground_truth_v1.csv"
STAGE_E_SUMMARY_PATH = STAGE_DE_ROOT / "results/b2a_external_synthetic_summary_v1.csv"
STAGE_E_ENVIRONMENT_PATH = STAGE_DE_ROOT / "results/b2a_external_synthetic_environment_summary_v1.csv"

FROZEN_STAGE_DE_HASHES = {
    STAGE_DE_DGP_PATH:
        "95884a97bda259755d751ac597b172c0ec17878d0767ad7279bb61df39b7cca0",
    STAGE_DE_QUERY_PATH:
        "d0c022193284c495751b7168d62db8206a397f02537e54ca2e165a0248b83d6b",
    STAGE_DE_PREDICTION_PATH:
        "3bf964bb7e90268a4f5e603d0b16a71be017ac8a2daa20f6d347614981d4e6b3",
}

def stage_de_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def stage_de_scientific_sha256(path: Path) -> str:
    if path == STAGE_DE_DGP_PATH:
        dgp = json.loads(path.read_text(encoding="utf-8"))
        payload = json.dumps(
            {"environments": dgp["environments"]},
            sort_keys=True, separators=(",", ":"), allow_nan=False,
        ).encode("utf-8")
        return hashlib.sha256(payload).hexdigest()
    return stage_de_sha256(path)

for frozen_path, frozen_hash in FROZEN_STAGE_DE_HASHES.items():
    assert frozen_path.exists()
    assert stage_de_scientific_sha256(frozen_path) == frozen_hash

for future_path in (STAGE_D_GT_PATH, STAGE_E_SUMMARY_PATH, STAGE_E_ENVIRONMENT_PATH):
    assert not future_path.exists()

stage_d_dgp = json.loads(STAGE_DE_DGP_PATH.read_text(encoding="utf-8"))
stage_d_queries = pd.read_csv(
    STAGE_DE_QUERY_PATH, float_precision="round_trip"
).sort_values(["environment_id", "query_id"]).reset_index(drop=True)
stage_d_predictions_for_hash_only = STAGE_DE_PREDICTION_PATH
stage_d_environment_specs = {
    environment["environment_id"]: environment
    for environment in stage_d_dgp["environments"]
}
assert list(stage_d_environment_specs) == [f"EXT{index}" for index in range(6)]
assert stage_d_queries.shape == (240, 25)
assert stage_d_queries[["environment_id", "query_id"]].duplicated().sum() == 0
assert not stage_d_queries.isna().any().any()

STAGE_D_DRAWS = 500
STAGE_D_STATE_COLUMNS = [f"x_{node:02d}" for node in range(20)]
STAGE_D_GT_COLUMNS = [
    "environment_id",
    "query_id",
    "tau_gt",
    "paired_diff_sd",
    "mc_se",
    "running_tau_100",
    "running_tau_250",
    "running_tau_500",
    "split_half_tau_first_250",
    "split_half_tau_second_250",
]

print("Frozen prediction and Stage A input hashes: PASS")
print("Stage D input boundary established before ground-truth generation.")


In [ ]:
# Implement the frozen paired Monte Carlo worlds so only the middle action differs.
def stage_d_synchronous_step(
    current_state: np.ndarray,
    actions: np.ndarray,
    noise: np.ndarray,
    coefficients: np.ndarray,
    intervention_targets: np.ndarray,
    intervention_strengths: np.ndarray,
    nonlinear: bool,
) -> np.ndarray:
    if nonlinear:
        parent_basis = 0.75 * current_state + 0.25 * np.square(current_state)
    else:
        parent_basis = current_state
    preactivation = (
        0.50 * current_state
        + parent_basis @ coefficients
        + noise
    )
    rows = np.arange(current_state.shape[0])
    action_targets = intervention_targets[actions]
    preactivation[rows, action_targets] += intervention_strengths[actions]
    return np.tanh(preactivation)

def stage_d_query_seed_sequences(environment_spec):
    provenance = environment_spec["component_seed_sequences"]["monte_carlo"]
    monte_carlo_parent = np.random.SeedSequence(
        entropy=provenance["entropy"],
        spawn_key=tuple(provenance["spawn_key"]),
        pool_size=provenance["pool_size"],
    )
    return monte_carlo_parent.spawn(40)

def stage_d_environment_ground_truth(environment_id: str) -> pd.DataFrame:
    environment_spec = stage_d_environment_specs[environment_id]
    coefficients = np.asarray(
        environment_spec["coefficient_matrix"], dtype=np.float64
    )
    intervention_targets = np.asarray(
        environment_spec["intervention_target_by_action"], dtype=np.int64
    )
    intervention_strengths = np.asarray(
        environment_spec["intervention_strength_by_action"], dtype=np.float64
    )
    action_probabilities = np.asarray(
        environment_spec["action_probabilities"], dtype=np.float64
    )
    sigma = float(environment_spec["noise_sigma"])
    nonlinear = bool(environment_spec["mechanism"]["nonlinear"])
    query_seed_sequences = stage_d_query_seed_sequences(environment_spec)
    environment_queries = stage_d_queries.loc[
        stage_d_queries["environment_id"] == environment_id
    ].sort_values("query_id")
    assert environment_queries["query_id"].tolist() == list(range(40))

    records = []
    for query in environment_queries.itertuples(index=False):
        query_id = int(query.query_id)
        intervention_action = int(query.intervention_action)
        reference_action = int(query.reference_action)
        target_construct = int(query.target_construct)
        conditioning_state = np.asarray(
            [getattr(query, column) for column in STAGE_D_STATE_COLUMNS],
            dtype=np.float64,
        )
        query_rng = np.random.Generator(
            np.random.PCG64(query_seed_sequences[query_id])
        )

        nuisance_previous = np.empty(STAGE_D_DRAWS, dtype=np.int64)
        nuisance_next = np.empty(STAGE_D_DRAWS, dtype=np.int64)
        transition_noise = np.empty(
            (STAGE_D_DRAWS, 3, 20), dtype=np.float64
        )
        for draw in range(STAGE_D_DRAWS):
            nuisance_previous[draw] = query_rng.choice(
                8, p=action_probabilities
            )
            nuisance_next[draw] = query_rng.choice(
                8, p=action_probabilities
            )
            transition_noise[draw] = query_rng.normal(
                0.0, sigma, size=(3, 20)
            )

        conditioned = np.repeat(
            conditioning_state.reshape(1, 20), STAGE_D_DRAWS, axis=0
        )
        state_t = stage_d_synchronous_step(
            conditioned,
            nuisance_previous,
            transition_noise[:, 0, :],
            coefficients,
            intervention_targets,
            intervention_strengths,
            nonlinear,
        )
        intervention_world_t1 = stage_d_synchronous_step(
            state_t,
            np.full(STAGE_D_DRAWS, intervention_action, dtype=np.int64),
            transition_noise[:, 1, :],
            coefficients,
            intervention_targets,
            intervention_strengths,
            nonlinear,
        )
        reference_world_t1 = stage_d_synchronous_step(
            state_t,
            np.full(STAGE_D_DRAWS, reference_action, dtype=np.int64),
            transition_noise[:, 1, :],
            coefficients,
            intervention_targets,
            intervention_strengths,
            nonlinear,
        )
        intervention_world_t2 = stage_d_synchronous_step(
            intervention_world_t1,
            nuisance_next,
            transition_noise[:, 2, :],
            coefficients,
            intervention_targets,
            intervention_strengths,
            nonlinear,
        )
        reference_world_t2 = stage_d_synchronous_step(
            reference_world_t1,
            nuisance_next,
            transition_noise[:, 2, :],
            coefficients,
            intervention_targets,
            intervention_strengths,
            nonlinear,
        )

        paired_differences = (
            intervention_world_t2[:, target_construct]
            - reference_world_t2[:, target_construct]
        )
        paired_diff_sd = float(paired_differences.std(ddof=1))
        tau_gt = float(paired_differences.mean())
        records.append({
            "environment_id": environment_id,
            "query_id": query_id,
            "tau_gt": tau_gt,
            "paired_diff_sd": paired_diff_sd,
            "mc_se": paired_diff_sd / math.sqrt(STAGE_D_DRAWS),
            "running_tau_100": float(paired_differences[:100].mean()),
            "running_tau_250": float(paired_differences[:250].mean()),
            "running_tau_500": float(paired_differences.mean()),
            "split_half_tau_first_250": float(
                paired_differences[:250].mean()
            ),
            "split_half_tau_second_250": float(
                paired_differences[250:].mean()
            ),
        })
    return pd.DataFrame.from_records(records, columns=STAGE_D_GT_COLUMNS)


In [ ]:
# Generate, validate, serialize, and hash all ground truth before Stage E can begin.
stage_d_ground_truth = pd.concat(
    [
        stage_d_environment_ground_truth(f"EXT{environment_index}")
        for environment_index in range(6)
    ],
    ignore_index=True,
)
assert stage_d_ground_truth.shape == (240, 10)
assert (
    stage_d_ground_truth.groupby("environment_id", sort=False).size().tolist()
    == [40] * 6
)
assert (
    stage_d_ground_truth[["environment_id", "query_id"]]
    .duplicated()
    .sum()
    == 0
)
assert np.isfinite(
    stage_d_ground_truth.drop(
        columns=["environment_id", "query_id"]
    ).to_numpy()
).all()
assert np.array_equal(
    stage_d_ground_truth["tau_gt"].to_numpy(),
    stage_d_ground_truth["running_tau_500"].to_numpy(),
)
assert np.allclose(
    stage_d_ground_truth["mc_se"].to_numpy(),
    stage_d_ground_truth["paired_diff_sd"].to_numpy()
    / math.sqrt(STAGE_D_DRAWS),
    rtol=0.0,
    atol=0.0,
)

stage_d_gt_buffer = io.StringIO()
stage_d_ground_truth.to_csv(
    stage_d_gt_buffer,
    index=False,
    lineterminator="\n",
    float_format=lambda value: repr(float(value)),
)
stage_d_gt_bytes = stage_d_gt_buffer.getvalue().encode("utf-8")
STAGE_D_GT_PATH.write_bytes(stage_d_gt_bytes)
stage_d_gt_sha256 = stage_de_sha256(STAGE_D_GT_PATH)
stage_d_gt_byte_size = STAGE_D_GT_PATH.stat().st_size
assert stage_de_sha256(STAGE_DE_PREDICTION_PATH) == FROZEN_STAGE_DE_HASHES[
    STAGE_DE_PREDICTION_PATH
]

stage_d_mc_summary = (
    stage_d_ground_truth.groupby("environment_id", sort=False)
    .agg(
        query_count=("tau_gt", "size"),
        tau_gt_min=("tau_gt", "min"),
        tau_gt_max=("tau_gt", "max"),
        tau_gt_mean=("tau_gt", "mean"),
        paired_diff_sd_mean=("paired_diff_sd", "mean"),
        mc_se_mean=("mc_se", "mean"),
        mc_se_max=("mc_se", "max"),
        running_100_max_abs_change=(
            "running_tau_100",
            lambda values: float(
                np.max(
                    np.abs(
                        values.to_numpy()
                        - stage_d_ground_truth.loc[
                            values.index, "tau_gt"
                        ].to_numpy()
                    )
                )
            ),
        ),
        running_250_max_abs_change=(
            "running_tau_250",
            lambda values: float(
                np.max(
                    np.abs(
                        values.to_numpy()
                        - stage_d_ground_truth.loc[
                            values.index, "tau_gt"
                        ].to_numpy()
                    )
                )
            ),
        ),
    )
    .reset_index()
)

print("Stage D ground truth finalized before evaluation.")
print(f"GT rows: {len(stage_d_ground_truth)}")
print(f"GT bytes: {stage_d_gt_byte_size}")
print(f"GT SHA-256: {stage_d_gt_sha256}")
display(stage_d_mc_summary)


In [ ]:
# Join frozen prediction and finalized ground truth keys to compute the prespecified metrics.
assert STAGE_D_GT_PATH.exists()
assert stage_de_sha256(STAGE_D_GT_PATH) == stage_d_gt_sha256
assert stage_de_sha256(STAGE_DE_PREDICTION_PATH) == FROZEN_STAGE_DE_HASHES[
    STAGE_DE_PREDICTION_PATH
]

stage_e_predictions = pd.read_csv(
    STAGE_DE_PREDICTION_PATH, float_precision="round_trip"
)
stage_e_ground_truth = pd.read_csv(
    STAGE_D_GT_PATH, float_precision="round_trip"
)
stage_e_evaluation = stage_e_predictions.merge(
    stage_e_ground_truth,
    on=["environment_id", "query_id"],
    how="inner",
    validate="one_to_one",
)
assert stage_e_evaluation.shape[0] == 240
assert (
    stage_e_evaluation[["environment_id", "query_id"]]
    .duplicated()
    .sum()
    == 0
)

stage_e_evaluation["error"] = (
    stage_e_evaluation["tau_hat"] - stage_e_evaluation["tau_gt"]
)
stage_e_evaluation["absolute_error"] = np.abs(
    stage_e_evaluation["error"]
)

def stage_e_exact_sign(values: np.ndarray) -> np.ndarray:
    return np.where(
        values > 0.0,
        "positive",
        np.where(values < 0.0, "negative", "zero"),
    )

stage_e_evaluation["prediction_sign"] = stage_e_exact_sign(
    stage_e_evaluation["tau_hat"].to_numpy()
)
stage_e_evaluation["gt_sign"] = stage_e_exact_sign(
    stage_e_evaluation["tau_gt"].to_numpy()
)
stage_e_evaluation["sign_agreement_row"] = (
    stage_e_evaluation["prediction_sign"]
    == stage_e_evaluation["gt_sign"]
)

STAGE_E_SIGN_CATEGORIES = ("negative", "zero", "positive")

def stage_e_correlation(frame: pd.DataFrame, method: str) -> float:
    prediction = frame["tau_hat"].to_numpy(dtype=np.float64)
    ground_truth = frame["tau_gt"].to_numpy(dtype=np.float64)
    if np.ptp(prediction) == 0.0 or np.ptp(ground_truth) == 0.0:
        return float("nan")
    if method == "pearson":
        return float(stats.pearsonr(prediction, ground_truth).statistic)
    if method == "spearman":
        return float(stats.spearmanr(prediction, ground_truth).statistic)
    raise ValueError(method)

def stage_e_metric_record(frame: pd.DataFrame) -> dict:
    error = frame["error"].to_numpy(dtype=np.float64)
    record = {
        "n_queries": int(len(frame)),
        "mae": float(np.mean(np.abs(error))),
        "rmse": float(np.sqrt(np.mean(np.square(error)))),
        "pearson": stage_e_correlation(frame, "pearson"),
        "spearman": stage_e_correlation(frame, "spearman"),
        "sign_agreement": float(frame["sign_agreement_row"].mean()),
        "prediction_min": float(frame["tau_hat"].min()),
        "prediction_max": float(frame["tau_hat"].max()),
        "ground_truth_min": float(frame["tau_gt"].min()),
        "ground_truth_max": float(frame["tau_gt"].max()),
        "mean_prediction": float(frame["tau_hat"].mean()),
        "mean_ground_truth": float(frame["tau_gt"].mean()),
        "prediction_zero_count": int((frame["prediction_sign"] == "zero").sum()),
        "ground_truth_zero_count": int((frame["gt_sign"] == "zero").sum()),
        "paired_diff_sd_mean": float(frame["paired_diff_sd"].mean()),
        "paired_diff_sd_median": float(frame["paired_diff_sd"].median()),
        "paired_diff_sd_max": float(frame["paired_diff_sd"].max()),
        "mc_se_mean": float(frame["mc_se"].mean()),
        "mc_se_median": float(frame["mc_se"].median()),
        "mc_se_max": float(frame["mc_se"].max()),
        "running_100_abs_change_mean": float(
            np.mean(np.abs(frame["running_tau_100"] - frame["tau_gt"]))
        ),
        "running_100_abs_change_max": float(
            np.max(np.abs(frame["running_tau_100"] - frame["tau_gt"]))
        ),
        "running_250_abs_change_mean": float(
            np.mean(np.abs(frame["running_tau_250"] - frame["tau_gt"]))
        ),
        "running_250_abs_change_max": float(
            np.max(np.abs(frame["running_tau_250"] - frame["tau_gt"]))
        ),
        "split_half_abs_difference_mean": float(
            np.mean(
                np.abs(
                    frame["split_half_tau_first_250"]
                    - frame["split_half_tau_second_250"]
                )
            )
        ),
        "split_half_abs_difference_max": float(
            np.max(
                np.abs(
                    frame["split_half_tau_first_250"]
                    - frame["split_half_tau_second_250"]
                )
            )
        ),
    }
    for gt_sign in STAGE_E_SIGN_CATEGORIES:
        for prediction_sign in STAGE_E_SIGN_CATEGORIES:
            record[
                f"sign_gt_{gt_sign}_pred_{prediction_sign}"
            ] = int(
                (
                    (frame["gt_sign"] == gt_sign)
                    & (frame["prediction_sign"] == prediction_sign)
                ).sum()
            )
    return record

stage_e_pooled_record = stage_e_metric_record(stage_e_evaluation)
stage_e_environment_records = []
for environment_id in [f"EXT{index}" for index in range(6)]:
    environment_frame = stage_e_evaluation.loc[
        stage_e_evaluation["environment_id"] == environment_id
    ]
    stage_e_environment_records.append({
        "environment_id": environment_id,
        **stage_e_metric_record(environment_frame),
    })

stage_e_sign_confusion = (
    stage_e_evaluation.groupby(
        ["gt_sign", "prediction_sign"], observed=True
    )
    .size()
    .reindex(
        pd.MultiIndex.from_product(
            [STAGE_E_SIGN_CATEGORIES, STAGE_E_SIGN_CATEGORIES],
            names=["gt_sign", "prediction_sign"],
        ),
        fill_value=0,
    )
    .rename("count")
    .reset_index()
)
print("Stage E joined frozen predictions and GT by query key only.")
display(stage_e_sign_confusion)


In [ ]:
# Apply the frozen stratified bootstrap to quantify pooled metric uncertainty.
STAGE_E_BOOTSTRAP_REPLICATES = 10_000
STAGE_E_BOOTSTRAP_SEED = 20260909
stage_e_bootstrap_rng = np.random.Generator(
    np.random.PCG64(STAGE_E_BOOTSTRAP_SEED)
)
stage_e_environment_indices = {
    environment_id: stage_e_evaluation.index[
        stage_e_evaluation["environment_id"] == environment_id
    ].to_numpy()
    for environment_id in [f"EXT{index}" for index in range(6)]
}
assert all(len(indices) == 40 for indices in stage_e_environment_indices.values())

stage_e_bootstrap_values = np.empty(
    (STAGE_E_BOOTSTRAP_REPLICATES, 3), dtype=np.float64
)
for replicate in range(STAGE_E_BOOTSTRAP_REPLICATES):
    sampled_indices = np.concatenate([
        stage_e_bootstrap_rng.choice(
            indices, size=40, replace=True
        )
        for indices in stage_e_environment_indices.values()
    ])
    sampled = stage_e_evaluation.loc[sampled_indices]
    sampled_error = sampled["error"].to_numpy(dtype=np.float64)
    stage_e_bootstrap_values[replicate, 0] = np.mean(
        np.abs(sampled_error)
    )
    stage_e_bootstrap_values[replicate, 1] = np.sqrt(
        np.mean(np.square(sampled_error))
    )
    stage_e_bootstrap_values[replicate, 2] = np.mean(
        sampled["sign_agreement_row"].to_numpy(dtype=np.float64)
    )

stage_e_bootstrap_ci = np.percentile(
    stage_e_bootstrap_values, [2.5, 97.5], axis=0
)
stage_e_pooled_record.update({
    "mae_ci_lower": float(stage_e_bootstrap_ci[0, 0]),
    "mae_ci_upper": float(stage_e_bootstrap_ci[1, 0]),
    "rmse_ci_lower": float(stage_e_bootstrap_ci[0, 1]),
    "rmse_ci_upper": float(stage_e_bootstrap_ci[1, 1]),
    "sign_agreement_ci_lower": float(stage_e_bootstrap_ci[0, 2]),
    "sign_agreement_ci_upper": float(stage_e_bootstrap_ci[1, 2]),
    "bootstrap_replicates": STAGE_E_BOOTSTRAP_REPLICATES,
    "bootstrap_seed": STAGE_E_BOOTSTRAP_SEED,
    "bootstrap_design": "environment-stratified; 40 draws with replacement per environment",
})
stage_e_summary = pd.DataFrame([
    {"scope": "pooled", **stage_e_pooled_record}
])
stage_e_environment_summary = pd.DataFrame(stage_e_environment_records)
assert stage_e_summary.shape[0] == 1
assert stage_e_environment_summary.shape[0] == 6

print("Environment-stratified bootstrap: PASS")
print(f"Replicates: {STAGE_E_BOOTSTRAP_REPLICATES}")


In [ ]:
# Save and verify only the two frozen summaries while preserving prediction and GT hashes.
def stage_e_csv_bytes(frame: pd.DataFrame) -> bytes:
    buffer = io.StringIO()
    frame.to_csv(
        buffer,
        index=False,
        lineterminator="\n",
        float_format=lambda value: repr(float(value)),
    )
    return buffer.getvalue().encode("utf-8")

STAGE_E_SUMMARY_PATH.write_bytes(stage_e_csv_bytes(stage_e_summary))
STAGE_E_ENVIRONMENT_PATH.write_bytes(
    stage_e_csv_bytes(stage_e_environment_summary)
)
stage_e_summary_sha256 = stage_de_sha256(STAGE_E_SUMMARY_PATH)
stage_e_environment_sha256 = stage_de_sha256(STAGE_E_ENVIRONMENT_PATH)

persisted_stage_e_summary = pd.read_csv(
    STAGE_E_SUMMARY_PATH, float_precision="round_trip"
)
persisted_stage_e_environment = pd.read_csv(
    STAGE_E_ENVIRONMENT_PATH, float_precision="round_trip"
)
pd.testing.assert_frame_equal(
    persisted_stage_e_summary,
    stage_e_summary,
    check_exact=True,
    check_dtype=False,
)
pd.testing.assert_frame_equal(
    persisted_stage_e_environment,
    stage_e_environment_summary,
    check_exact=True,
    check_dtype=False,
)
assert stage_de_sha256(STAGE_D_GT_PATH) == stage_d_gt_sha256
assert stage_de_sha256(STAGE_DE_PREDICTION_PATH) == FROZEN_STAGE_DE_HASHES[
    STAGE_DE_PREDICTION_PATH
]
assert np.isfinite(
    stage_e_summary[
        ["mae", "rmse", "pearson", "spearman", "sign_agreement"]
    ].to_numpy()
).all()
assert np.isfinite(
    stage_e_environment_summary[
        ["mae", "rmse", "pearson", "spearman", "sign_agreement"]
    ].to_numpy()
).all()

print("Stage E canonical summaries finalized.")
print(f"Summary SHA-256: {stage_e_summary_sha256}")
print(f"Environment summary SHA-256: {stage_e_environment_sha256}")
print("Frozen prediction hash remained byte-identical.")
display(stage_e_summary)
display(
    stage_e_environment_summary[
        [
            "environment_id", "n_queries", "mae", "rmse",
            "pearson", "spearman", "sign_agreement",
            "ground_truth_min", "ground_truth_max",
            "mc_se_mean", "mc_se_max",
        ]
    ]
)
print("STOP: Frozen Stage D and Stage E complete; no tuning or extra experiment executed.")


## Secondary Post-Validation Baselines

These analyses were added after the external-synthetic ground truth had already been generated and inspected. They are comparative sensitivity/reference analyses, not blinded or prospectively frozen independent-validation evidence. Exactly two baselines are evaluated: `ZERO` and `POOLED_RIDGE`. Neither baseline is tuned, and the frozen B2A estimator and predictions remain unchanged.


In [ ]:
# Verify every canonical input hash and load only the frozen data needed for the two baselines.
from __future__ import annotations

import hashlib
import io
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import Ridge

BASELINE_ROOT = Path.cwd()
if BASELINE_ROOT.name == "notebooks":
    BASELINE_ROOT = BASELINE_ROOT.parent

BASELINE_TRAINING_PATH = BASELINE_ROOT / "results/b2a_external_synthetic_training_v1.npz"
BASELINE_QUERY_PATH = BASELINE_ROOT / "results/b2a_external_synthetic_queries_v1.csv"
BASELINE_GT_PATH = BASELINE_ROOT / "results/b2a_external_synthetic_ground_truth_v1.csv"
BASELINE_B2A_PATH = BASELINE_ROOT / "results/b2a_external_synthetic_predictions_v1.csv"
BASELINE_DGP_PATH = BASELINE_ROOT / "results/b2a_external_synthetic_dgp_spec_v1.json"
BASELINE_MANIFEST_PATH = BASELINE_ROOT / "results/b2a_external_synthetic_manifest_v1.json"
BASELINE_SUMMARY_PATH = BASELINE_ROOT / "results/b2a_external_synthetic_summary_v1.csv"
BASELINE_ENVIRONMENT_SUMMARY_PATH = BASELINE_ROOT / "results/b2a_external_synthetic_environment_summary_v1.csv"
BASELINE_COMPARISON_PATH = BASELINE_ROOT / "results/b2a_external_synthetic_baseline_comparison_v1.csv"
BASELINE_ENVIRONMENT_PATH = BASELINE_ROOT / "results/b2a_external_synthetic_baseline_environment_v1.csv"

FROZEN_PRE_BASELINE_HASHES = {
    BASELINE_QUERY_PATH: "d0c022193284c495751b7168d62db8206a397f02537e54ca2e165a0248b83d6b",
    BASELINE_GT_PATH: "735468669caf262be233de69de4afbbe46a2caaa8a922892486e5d512a3f25e5",
    BASELINE_B2A_PATH: "3bf964bb7e90268a4f5e603d0b16a71be017ac8a2daa20f6d347614981d4e6b3",
    BASELINE_SUMMARY_PATH: "74dce3b91239e683534518d2c170dec5969f49f9f5039e8d40f0cffe99cd7fd5",
    BASELINE_ENVIRONMENT_SUMMARY_PATH: "0931f97df94d0cd6716116a08ee0b6fb25b94f0b3b0b0228c79fb90195c5be3c",
}

FROZEN_PRE_BASELINE_SCIENTIFIC_HASHES = {
    BASELINE_DGP_PATH: "95884a97bda259755d751ac597b172c0ec17878d0767ad7279bb61df39b7cca0",
    BASELINE_TRAINING_PATH: "f215ab76c2106a0971f9353630b47c430477c105e858cf79adaff6fb806bd838",
}

def baseline_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def baseline_scientific_sha256(path: Path) -> str:
    if path == BASELINE_DGP_PATH:
        dgp = json.loads(path.read_text(encoding="utf-8"))
        payload = json.dumps(
            {"environments": dgp["environments"]},
            sort_keys=True, separators=(",", ":"), allow_nan=False,
        ).encode("utf-8")
        return hashlib.sha256(payload).hexdigest()
    digest = hashlib.sha256()
    with np.load(path, allow_pickle=False) as arrays:
        for key in sorted(arrays.files):
            array = np.ascontiguousarray(arrays[key])
            shape = json.dumps(list(array.shape), separators=(",", ":")).encode()
            for token in (key.encode(), array.dtype.str.encode(), shape):
                digest.update(len(token).to_bytes(8, "big"))
                digest.update(token)
            digest.update(array.tobytes(order="C"))
    return digest.hexdigest()

for frozen_path, expected_hash in FROZEN_PRE_BASELINE_HASHES.items():
    assert frozen_path.exists()
    assert baseline_sha256(frozen_path) == expected_hash
for frozen_path, expected_hash in FROZEN_PRE_BASELINE_SCIENTIFIC_HASHES.items():
    assert frozen_path.exists()
    assert baseline_scientific_sha256(frozen_path) == expected_hash
baseline_manifest = json.loads(BASELINE_MANIFEST_PATH.read_text(encoding="utf-8"))
assert baseline_manifest["schema_version"] == "b2a_external_synthetic_manifest_v1"
for future_path in (BASELINE_COMPARISON_PATH, BASELINE_ENVIRONMENT_PATH):
    assert not future_path.exists()

baseline_training = dict(np.load(BASELINE_TRAINING_PATH, allow_pickle=False))
baseline_queries = pd.read_csv(
    BASELINE_QUERY_PATH, float_precision="round_trip"
).sort_values(["environment_id", "query_id"]).reset_index(drop=True)
baseline_ground_truth = pd.read_csv(
    BASELINE_GT_PATH, float_precision="round_trip"
).sort_values(["environment_id", "query_id"]).reset_index(drop=True)
baseline_b2a = pd.read_csv(
    BASELINE_B2A_PATH, float_precision="round_trip"
).sort_values(["environment_id", "query_id"]).reset_index(drop=True)

BASELINE_ENVIRONMENTS = [f"EXT{index}" for index in range(6)]
BASELINE_STATE_COLUMNS = [f"x_{node:02d}" for node in range(20)]
assert baseline_queries.shape == (240, 25)
assert baseline_ground_truth.shape == (240, 10)
assert baseline_b2a.shape == (240, 8)
assert baseline_queries.groupby("environment_id", sort=False).size().tolist() == [40] * 6
assert baseline_queries[["environment_id", "query_id"]].duplicated().sum() == 0
assert baseline_ground_truth[["environment_id", "query_id"]].duplicated().sum() == 0
assert baseline_b2a[["environment_id", "query_id"]].duplicated().sum() == 0

print("Frozen pre-baseline artifact hashes: PASS")
print("Ground truth was already observed; these baselines are post-validation comparisons.")


In [ ]:
# Generate only ZERO and six fixed pooled Ridge predictions to provide the authorized references.
baseline_prediction_frames = [
    baseline_b2a[["environment_id", "query_id", "tau_hat"]]
    .assign(method="B2A")
]
baseline_prediction_frames.append(
    baseline_queries[["environment_id", "query_id"]]
    .assign(tau_hat=0.0, method="ZERO")
)

pooled_prediction_records = []
pooled_fit_records = []
for environment_id in BASELINE_ENVIRONMENTS:
    states = baseline_training[f"{environment_id}_states"]
    actions = baseline_training[f"{environment_id}_actions"]
    assert states.shape == (400, 120, 20)
    assert actions.shape == (400, 119)

    aligned_states = states[:, 10:117, :].reshape(-1, 20)
    aligned_actions = actions[:, 11:118].reshape(-1).astype(np.int64)
    aligned_targets = states[:, 13:120, :].reshape(-1, 20)
    action_one_hot = np.eye(8, dtype=np.float64)[aligned_actions]
    pooled_features = np.concatenate([aligned_states, action_one_hot], axis=1)
    assert aligned_states.shape == (42_800, 20)
    assert aligned_actions.shape == (42_800,)
    assert aligned_targets.shape == (42_800, 20)
    assert pooled_features.shape == (42_800, 28)
    assert np.array_equal(np.unique(aligned_actions), np.arange(8))

    pooled_model = Ridge(alpha=1.0)
    pooled_model.fit(pooled_features, aligned_targets)
    pooled_fit_records.append({
        "environment_id": environment_id,
        "aligned_rows": len(pooled_features),
        "input_features": pooled_features.shape[1],
        "outputs": aligned_targets.shape[1],
    })

    environment_queries = baseline_queries.loc[
        baseline_queries["environment_id"] == environment_id
    ].sort_values("query_id")
    query_states = environment_queries[BASELINE_STATE_COLUMNS].to_numpy(
        dtype=np.float64
    )
    intervention_actions = environment_queries[
        "intervention_action"
    ].to_numpy(dtype=np.int64)
    reference_actions = environment_queries[
        "reference_action"
    ].to_numpy(dtype=np.int64)
    intervention_inputs = np.concatenate(
        [query_states, np.eye(8, dtype=np.float64)[intervention_actions]],
        axis=1,
    )
    reference_inputs = np.concatenate(
        [query_states, np.eye(8, dtype=np.float64)[reference_actions]],
        axis=1,
    )
    intervention_predictions = pooled_model.predict(intervention_inputs)
    reference_predictions = pooled_model.predict(reference_inputs)

    for row_index, query in enumerate(environment_queries.itertuples(index=False)):
        target = int(query.target_construct)
        pooled_prediction_records.append({
            "environment_id": environment_id,
            "query_id": int(query.query_id),
            "tau_hat": float(
                intervention_predictions[row_index, target]
                - reference_predictions[row_index, target]
            ),
            "method": "POOLED_RIDGE",
        })

baseline_prediction_frames.append(pd.DataFrame.from_records(pooled_prediction_records))
baseline_predictions = pd.concat(baseline_prediction_frames, ignore_index=True)
baseline_predictions = baseline_predictions[
    ["method", "environment_id", "query_id", "tau_hat"]
]
baseline_fit_summary = pd.DataFrame.from_records(pooled_fit_records)
assert set(baseline_predictions["method"]) == {"B2A", "ZERO", "POOLED_RIDGE"}
assert baseline_predictions.groupby("method").size().to_dict() == {
    "B2A": 240, "POOLED_RIDGE": 240, "ZERO": 240
}
assert baseline_predictions[
    ["method", "environment_id", "query_id"]
].duplicated().sum() == 0
assert np.isfinite(baseline_predictions["tau_hat"].to_numpy()).all()
assert baseline_fit_summary.shape == (6, 4)
assert (baseline_fit_summary["aligned_rows"] == 42_800).all()
assert (baseline_fit_summary["input_features"] == 28).all()
assert (baseline_fit_summary["outputs"] == 20).all()

print("Exactly 240 predictions per method: PASS")
print("Exactly six untuned pooled Ridge models fitted: PASS")
display(baseline_fit_summary)


In [ ]:
# Evaluate all three methods identically and add the prespecified descriptive zero-GT diagnostics.
baseline_evaluation = baseline_predictions.merge(
    baseline_ground_truth[["environment_id", "query_id", "tau_gt"]],
    on=["environment_id", "query_id"],
    how="left",
    validate="many_to_one",
)
assert len(baseline_evaluation) == 720
assert baseline_evaluation["tau_gt"].notna().all()
baseline_evaluation["error"] = (
    baseline_evaluation["tau_hat"] - baseline_evaluation["tau_gt"]
)
baseline_evaluation["absolute_error"] = np.abs(baseline_evaluation["error"])
baseline_evaluation["prediction_sign"] = np.sign(baseline_evaluation["tau_hat"])
baseline_evaluation["gt_sign"] = np.sign(baseline_evaluation["tau_gt"])
baseline_evaluation["exact_sign_match"] = (
    baseline_evaluation["prediction_sign"] == baseline_evaluation["gt_sign"]
)

def baseline_correlation(frame: pd.DataFrame, method: str) -> float:
    prediction = frame["tau_hat"].to_numpy(dtype=np.float64)
    ground_truth = frame["tau_gt"].to_numpy(dtype=np.float64)
    if np.ptp(prediction) == 0.0 or np.ptp(ground_truth) == 0.0:
        return float("nan")
    if method == "pearson":
        return float(stats.pearsonr(prediction, ground_truth).statistic)
    if method == "spearman":
        return float(stats.spearmanr(prediction, ground_truth).statistic)
    raise ValueError(method)

def baseline_metric_record(frame: pd.DataFrame, include_zero_diagnostics: bool) -> dict:
    error = frame["error"].to_numpy(dtype=np.float64)
    prediction_is_constant = np.ptp(
        frame["tau_hat"].to_numpy(dtype=np.float64)
    ) == 0.0
    record = {
        "n_queries": int(len(frame)),
        "mae": float(np.mean(np.abs(error))),
        "rmse": float(np.sqrt(np.mean(np.square(error)))),
        "pearson": baseline_correlation(frame, "pearson"),
        "spearman": baseline_correlation(frame, "spearman"),
        "correlation_note": (
            "undefined: predictions are constant"
            if prediction_is_constant else "defined"
        ),
        "exact_sign_agreement": float(frame["exact_sign_match"].mean()),
    }
    if include_zero_diagnostics:
        gt_zero = frame.loc[frame["gt_sign"] == 0.0]
        gt_nonzero = frame.loc[frame["gt_sign"] != 0.0]
        assert len(gt_zero) > 0 and len(gt_nonzero) > 0
        record.update({
            "gt_zero_count": int(len(gt_zero)),
            "gt_nonzero_count": int(len(gt_nonzero)),
            "gt_zero_mae": float(gt_zero["absolute_error"].mean()),
            "gt_nonzero_mae": float(gt_nonzero["absolute_error"].mean()),
            "gt_nonzero_sign_agreement": float(
                gt_nonzero["exact_sign_match"].mean()
            ),
        })
    return record

BASELINE_METHOD_ORDER = ["B2A", "ZERO", "POOLED_RIDGE"]
comparison_records = []
environment_comparison_records = []
for method in BASELINE_METHOD_ORDER:
    method_frame = baseline_evaluation.loc[
        baseline_evaluation["method"] == method
    ]
    comparison_records.append({
        "method": method,
        **baseline_metric_record(method_frame, include_zero_diagnostics=True),
    })
    for environment_id in BASELINE_ENVIRONMENTS:
        environment_frame = method_frame.loc[
            method_frame["environment_id"] == environment_id
        ]
        environment_comparison_records.append({
            "environment_id": environment_id,
            "method": method,
            **baseline_metric_record(
                environment_frame, include_zero_diagnostics=False
            ),
        })

baseline_comparison = pd.DataFrame.from_records(comparison_records)
baseline_environment_comparison = pd.DataFrame.from_records(
    environment_comparison_records
)
zero_pooled = baseline_comparison.loc[
    baseline_comparison["method"] == "ZERO"
].iloc[0]
for metric in ("mae", "rmse"):
    baseline_comparison[f"{metric}_reduction_vs_zero"] = (
        zero_pooled[metric] - baseline_comparison[metric]
    ) / zero_pooled[metric]
for environment_id in BASELINE_ENVIRONMENTS:
    environment_mask = (
        baseline_environment_comparison["environment_id"] == environment_id
    )
    zero_environment = baseline_environment_comparison.loc[
        environment_mask
        & (baseline_environment_comparison["method"] == "ZERO")
    ].iloc[0]
    for metric in ("mae", "rmse"):
        baseline_environment_comparison.loc[
            environment_mask, f"{metric}_reduction_vs_zero"
        ] = (
            zero_environment[metric]
            - baseline_environment_comparison.loc[environment_mask, metric]
        ) / zero_environment[metric]

assert baseline_comparison.shape == (3, 15)
assert baseline_environment_comparison.shape == (18, 11)
assert baseline_comparison["gt_zero_count"].tolist() == [158] * 3
assert baseline_comparison["gt_nonzero_count"].tolist() == [82] * 3
assert baseline_comparison.loc[
    baseline_comparison["method"] == "ZERO", "exact_sign_agreement"
].iat[0] == 158 / 240
assert baseline_comparison.loc[
    baseline_comparison["method"] == "ZERO", "gt_nonzero_sign_agreement"
].iat[0] == 0.0

print("Identical frozen-GT evaluation for all three methods: PASS")
display(baseline_comparison)


In [ ]:
# Freeze only the two concise comparison tables and enforce the allowed undefined-correlation NaNs.
def baseline_csv_bytes(frame: pd.DataFrame) -> bytes:
    buffer = io.StringIO()
    frame.to_csv(
        buffer,
        index=False,
        lineterminator="\n",
        na_rep="NaN",
        float_format=lambda value: repr(float(value)),
    )
    return buffer.getvalue().encode("utf-8")

BASELINE_COMPARISON_PATH.write_bytes(baseline_csv_bytes(baseline_comparison))
BASELINE_ENVIRONMENT_PATH.write_bytes(
    baseline_csv_bytes(baseline_environment_comparison)
)
baseline_comparison_sha256 = baseline_sha256(BASELINE_COMPARISON_PATH)
baseline_environment_sha256 = baseline_sha256(BASELINE_ENVIRONMENT_PATH)

persisted_baseline_comparison = pd.read_csv(
    BASELINE_COMPARISON_PATH, float_precision="round_trip"
)
persisted_baseline_environment = pd.read_csv(
    BASELINE_ENVIRONMENT_PATH, float_precision="round_trip"
)
pd.testing.assert_frame_equal(
    persisted_baseline_comparison,
    baseline_comparison,
    check_exact=True,
    check_dtype=False,
)
pd.testing.assert_frame_equal(
    persisted_baseline_environment,
    baseline_environment_comparison,
    check_exact=True,
    check_dtype=False,
)

comparison_nan_locations = set(
    zip(*np.where(persisted_baseline_comparison.isna().to_numpy()))
)
expected_comparison_nans = {
    (
        int(persisted_baseline_comparison.index[
            persisted_baseline_comparison["method"] == "ZERO"
        ][0]),
        int(persisted_baseline_comparison.columns.get_loc(column)),
    )
    for column in ("pearson", "spearman")
}
assert comparison_nan_locations == expected_comparison_nans

environment_nan_locations = set(
    zip(*np.where(persisted_baseline_environment.isna().to_numpy()))
)
expected_environment_nans = {
    (int(row_index), int(persisted_baseline_environment.columns.get_loc(column)))
    for row_index in persisted_baseline_environment.index[
        persisted_baseline_environment["method"] == "ZERO"
    ]
    for column in ("pearson", "spearman")
}
assert environment_nan_locations == expected_environment_nans

for frozen_path, expected_hash in FROZEN_PRE_BASELINE_HASHES.items():
    assert baseline_sha256(frozen_path) == expected_hash
for frozen_path, expected_hash in FROZEN_PRE_BASELINE_SCIENTIFIC_HASHES.items():
    assert baseline_scientific_sha256(frozen_path) == expected_hash
assert set(persisted_baseline_comparison["method"]) == {
    "B2A", "ZERO", "POOLED_RIDGE"
}
assert set(persisted_baseline_environment["method"]) == {
    "B2A", "ZERO", "POOLED_RIDGE"
}

print(f"Comparison SHA-256: {baseline_comparison_sha256}")
print(f"Environment comparison SHA-256: {baseline_environment_sha256}")
print("All prior canonical artifacts remained byte-identical.")
print("ZERO Pearson and Spearman are NaN because its predictions are constant.")


In [ ]:
# Display the final post-validation comparisons and stop without tuning or modifying B2A.
pooled_display_columns = [
    "method", "n_queries", "mae", "rmse", "pearson", "spearman",
    "exact_sign_agreement", "mae_reduction_vs_zero",
    "rmse_reduction_vs_zero",
]
environment_display_columns = [
    "environment_id", "method", "n_queries", "mae", "rmse",
    "pearson", "spearman", "exact_sign_agreement",
    "mae_reduction_vs_zero", "rmse_reduction_vs_zero",
]
zero_gt_display_columns = [
    "method", "gt_zero_count", "gt_nonzero_count", "gt_zero_mae",
    "gt_nonzero_mae", "gt_nonzero_sign_agreement",
]

display(baseline_comparison[pooled_display_columns])
display(baseline_environment_comparison[environment_display_columns])
display(baseline_comparison[zero_gt_display_columns])
print("ZERO correlations are undefined (NaN) because all ZERO predictions are constant.")
print("These are post-validation sensitivity/reference analyses; GT was already observed.")
print("No baseline was tuned, and frozen B2A predictions remain unchanged.")
print("STOP: exactly two secondary baselines completed; no additional model tested.")


## Consolidated Draft-2 reviewer diagnostics

These are post-hoc descriptive reviewer-response diagnostics. No epsilon was
introduced, and no model was retrained, redesigned, or tuned. The canonical
per-query file contains B2A predictions but not the secondary Pooled Ridge
predictions. Consequently, Pooled Ridge quantities recoverable from its frozen
aggregate table are reported, while unavailable per-query quantities remain
missing rather than being recreated by refitting.

The temporal audit verdict is: **Alignment is correct; manuscript explanation
was insufficient.** `A[t+1]` is omitted from estimator features but is
marginalized under the frozen state-independent action policy and shared across
paired Monte Carlo worlds in the ground-truth estimand.


In [ ]:
# This cell loads exact-zero diagnostics because tied GT effects require separate descriptive accounting.
DRAFT2_ZERO_NONZERO_PATH = STAGE_DE_ROOT / "results/draft2_reviewer_zero_nonzero_v1.csv"
DRAFT2_ZERO_COMPOSITION_PATH = STAGE_DE_ROOT / "results/draft2_reviewer_zero_composition_by_environment_v1.csv"
DRAFT2_ZERO_PREDICTIONS_PATH = STAGE_DE_ROOT / "results/draft2_reviewer_zero_effect_predictions_v1.csv"
draft2_zero_nonzero = pd.read_csv(DRAFT2_ZERO_NONZERO_PATH)
draft2_zero_composition = pd.read_csv(DRAFT2_ZERO_COMPOSITION_PATH)
draft2_zero_predictions = pd.read_csv(DRAFT2_ZERO_PREDICTIONS_PATH)
assert draft2_zero_nonzero.groupby("estimator")["stratum"].nunique().eq(2).all()
assert draft2_zero_composition["exact_zero_count"].sum() == 158
assert draft2_zero_composition["nonzero_count"].sum() == 82
assert draft2_zero_predictions.loc[draft2_zero_predictions["estimator"] == "B2A", "predicted_exact_zero_count"].iat[0] == 0
display(draft2_zero_nonzero)
display(draft2_zero_composition)
display(draft2_zero_predictions)


In [ ]:
# This cell verifies temporal alignment and action overlap because the audit must trace the frozen implementation rather than rerun it.
DRAFT2_ALIGNMENT_PATH = STAGE_DE_ROOT / "results/draft2_reviewer_external_alignment_audit_v1.json"
draft2_alignment = json.loads(DRAFT2_ALIGNMENT_PATH.read_text())
assert draft2_alignment["alignment_status"] == "correct"
assert draft2_alignment["conditioning_state_index"] == "X[t-1]"
assert draft2_alignment["intervention_index"] == "A[t]"
assert draft2_alignment["target_index"] == "X[t+2,Y]"
assert draft2_alignment["paired_world_coupling"]["nuisance_actions_shared"]
assert draft2_alignment["paired_world_coupling"]["exogenous_noise_all_three_transitions_shared"]
assert draft2_alignment["action_assignment"]["state_independent"]
assert draft2_alignment["action_assignment"]["all_actions_support_count_ge_100"]
display(pd.DataFrame(draft2_alignment["action_assignment"]["environment_support_audit"]))
print(draft2_alignment["required_manuscript_statement"])


In [ ]:
# This cell displays scale and rank diagnostics because raw errors are not directly comparable across benchmark scales.
draft2_scale = pd.read_csv(STAGE_DE_ROOT / "results/draft2_reviewer_external_scale_audit_v1.csv")
draft2_rank = pd.read_csv(STAGE_DE_ROOT / "results/draft2_reviewer_external_rank_diagnostic_v1.csv")
assert set(draft2_scale["benchmark"]) == {"LOCAL_NEURIPS_DEVELOPMENT", "EXTERNAL_SYNTHETIC"}
assert draft2_rank["environment_id"].tolist() == [f"EXT{i}" for i in range(6)]
display(draft2_scale)
display(draft2_rank)


In [ ]:
# This cell rechecks frozen external hashes because audit-only diagnostics must leave prior evidence byte-identical.
FROZEN_EXTERNAL_HASHES_DRAFT2 = {
    "b2a_external_synthetic_predictions_v1.csv": "3bf964bb7e90268a4f5e603d0b16a71be017ac8a2daa20f6d347614981d4e6b3",
    "b2a_external_synthetic_ground_truth_v1.csv": "735468669caf262be233de69de4afbbe46a2caaa8a922892486e5d512a3f25e5",
    "b2a_external_synthetic_summary_v1.csv": "74dce3b91239e683534518d2c170dec5969f49f9f5039e8d40f0cffe99cd7fd5",
    "b2a_external_synthetic_environment_summary_v1.csv": "0931f97df94d0cd6716116a08ee0b6fb25b94f0b3b0b0228c79fb90195c5be3c",
    "b2a_external_synthetic_baseline_comparison_v1.csv": "bdae7fe961ee47e39f46cacb0fb71195c2e379caaec9ab0348eefc81e4a0bde0",
    "b2a_external_synthetic_baseline_environment_v1.csv": "b15fa3a49ecf0eeb6d1f09328162615a980132a1da8807f5ac7aa307a2be4cbc",
}
for filename, expected_hash in FROZEN_EXTERNAL_HASHES_DRAFT2.items():
    assert stage_de_sha256(STAGE_DE_ROOT / "results" / filename) == expected_hash
print("All frozen external-synthetic outputs remain byte-identical.")
